# Lasso Regression

Replication Portfolio as a regularized least-squares regression problem with an $\ell_1$ penalty (LASSO).

- **Train:** Jan 2020 – June 2025
- **Holdout Test:** July 2025 – Dec 2025


In [1]:
import os
import glob
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.express as px
import plotly.graph_objects as go
from sklearn.linear_model import Lasso
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

## 1. Data Loading

In [2]:
data_dir = '../data'
constituents_dir = os.path.join(data_dir, 'sp500 constituents')
benchmark_path = os.path.join(data_dir, '^GSPC.csv')

def load_data():
    # Load Benchmark
    print("Loading benchmark data...")
    df_bench = pd.read_csv(benchmark_path)
    df_bench['Date'] = pd.to_datetime(df_bench['Date'])
    df_bench = df_bench.sort_values('Date').set_index('Date')
    df_bench = df_bench.loc[~df_bench.index.duplicated(keep='last')]
    benchmark_prices = df_bench[['Close']].rename(columns={'Close': '^GSPC'})
    
    # Load Constituents
    print("Loading constituents data...")
    all_files = glob.glob(os.path.join(constituents_dir, '*.csv'))
    
    stock_prices_list = []
    
    for file in all_files:
        ticker = os.path.splitext(os.path.basename(file))[0]

        df_stock = pd.read_csv(file)
        df_stock['Date'] = pd.to_datetime(df_stock['Date'])
        df_stock = df_stock.sort_values('Date').set_index('Date')
        df_stock = df_stock.loc[~df_stock.index.duplicated(keep='last')]
            
        # Use Adj Close if available, else Close
        if 'Adj Close' in df_stock.columns:
            price_col = 'Adj Close' 
        else:
            price_col = 'Close'
        df_stock = df_stock[[price_col]].rename(columns={price_col: ticker})
        stock_prices_list.append(df_stock)
            
    # Concatenate all series along the Date index
    df_stocks = pd.concat(stock_prices_list, axis=1)
    
    # Merge with benchmark
    df_all = benchmark_prices.join(df_stocks, how='outer')
    
    # Forward fill missing values to handle non-trading days differences
    df_all = df_all.ffill()
    
    # Drop rows before Jan 2020 if any
    df_all = df_all[df_all.index >= '2020-01-01']
    
    # Drop columns that have excessive missing values (> 5% missing after start)
    missing_pct = df_all.isnull().mean()
    dropped_cols = missing_pct[missing_pct > 0.05].index
    print(f"Dropping {len(dropped_cols)} stocks due to missing data (>5% missing).")
    df_all = df_all.drop(columns=dropped_cols)
    
    # Drop any remaining NaNs 
    df_all = df_all.dropna()
    
    print(f"Final dataset shape: {df_all.shape}")
    return df_all

df_prices = load_data()
df_prices.head()


Loading benchmark data...
Loading constituents data...
Dropping 24 stocks due to missing data (>5% missing).
Final dataset shape: (1492, 560)


,^GSPC,A,AAL,AAP,AAPL,ABBV,ABT,ACGL,ACN,ADBE,...,XEL,XOM,XRAY,XYL,XYZ,YUM,ZBH,ZBRA,ZION,ZTS
Date,,,,,,,,,,,,,,,,,,,,,
2020-03-19,2409.389893,63.042397,10.29,76.012421,59.200542,55.706875,66.955215,25.731314,145.760696,307.510010,...,48.526421,26.432968,31.149555,60.207314,40.000000,56.204372,82.418640,185.070007,22.301153,97.001999
2020-03-20,2304.919922,63.713478,10.38,66.339561,55.442150,53.950867,61.113499,24.694834,137.888641,295.339996,...,42.367516,25.135504,29.117863,55.938820,38.090000,52.146164,79.496941,179.380005,20.934437,95.684357
2020-03-23,2237.399902,60.751183,10.25,69.407654,54.264336,50.564243,56.458073,22.117901,132.140991,307.269989,...,41.840324,24.145132,28.301638,52.903236,40.009998,50.745556,74.716759,170.720001,19.706852,87.835464
2020-03-24,2447.330078,64.231186,13.92,76.374931,59.708420,52.908218,62.632328,24.343002,144.243240,310.000000,...,45.229385,27.208378,29.996189,57.909611,46.310001,62.543079,86.357414,180.550003,20.607084,98.357521
2020-03-25,2475.560059,66.148529,15.39,79.902786,59.379501,53.237480,63.584976,24.457109,139.617554,305.910004,...,46.484608,28.628685,31.051954,59.646893,52.389999,65.425133,89.528824,186.229996,21.155401,102.575836


## 2. Data and Target


In [3]:
def compute_log_returns(df):
    return np.log(df / df.shift(1)).dropna()

df_returns = compute_log_returns(df_prices)

# Extract y and X
y = df_returns['^GSPC']
X = df_returns.drop(columns=['^GSPC'])

# Train: Jan 2020 - June 2025
# Holdout Test: July 2025 - Dec 2025
train_mask = (df_returns.index >= '2020-01-01') & (df_returns.index <= '2025-06-30')
test_mask = (df_returns.index >= '2025-07-01') & (df_returns.index <= '2025-12-31')

X_train, y_train = X.loc[train_mask], y.loc[train_mask]
X_test, y_test = X.loc[test_mask], y.loc[test_mask]

print(f"Training set: {X_train.shape[0]} days, {X_train.shape[1]} stocks")
print(f"Test set:     {X_test.shape[0]} days, {X_test.shape[1]} stocks")


Training set: 1326 days, 559 stocks
Test set:     128 days, 559 stocks


## 3. Lasso Regression & Cardinality Control

In [4]:
def evaluate_portfolio(R_p, R_b):
    # Tracking Error
    TE = np.std(R_p - R_b)
    # Information Ratio (mean(R_p - R_b) / std(R_p - R_b))
    if TE != 0:
        IR = np.mean(R_p - R_b) / TE 
    else:
        IR = 0
    return TE, IR

def fit_lasso_portfolio(X_train, y_train, alpha_val, max_k=50):
    # Fit Lasso
    lasso = Lasso(alpha=alpha_val, fit_intercept=False, max_iter=10000, random_state=42)
    lasso.fit(X_train, y_train)
    
    # Extract weights
    w = lasso.coef_
    
    # Identify non-zero weights
    non_zero_idx = np.where(w != 0)[0]
    
    # Cardinality control (k <= max_k)
    k = len(non_zero_idx)
    if k > max_k:
        # Keep top-k by absolute weight
        top_k_indices = np.argsort(np.abs(w))[-max_k:]
        selected_idx = top_k_indices
        w_selected = w[selected_idx]
        k = max_k
    else:
        selected_idx = non_zero_idx
        w_selected = w[selected_idx]
        
    # Normalize weights to sum to 1
    if np.sum(w_selected) != 0:
        w_norm = w_selected / np.sum(w_selected)
    else:
        w_norm = w_selected
        
    selected_tickers = X_train.columns[selected_idx].tolist()
    return selected_tickers, w_norm, k

# Find a good alpha to hit close to k=50
# Sweep alphas to plot Sparsity vs TE and choose one for our final portfolio
alphas = np.logspace(-6, -3, 50)
results = []

for idx, a in enumerate(alphas):
    selected_tickers, w_norm, k = fit_lasso_portfolio(X_train, y_train, alpha_val=a, max_k=500) # Unbounded to see actual sparsity
    
    if k == 0:
        continue
        
    X_train_sel = X_train[selected_tickers]
    R_p_train = X_train_sel @ w_norm
    TE_train, IR_train = evaluate_portfolio(R_p_train, y_train)
    
    X_test_sel = X_test[selected_tickers]
    R_p_test = X_test_sel @ w_norm
    TE_test, IR_test = evaluate_portfolio(R_p_test, y_test)
    
    results.append({
        'alpha': a,
        'k': k,
        'TE_train': TE_train,
        'IR_train': IR_train,
        'TE_test': TE_test,
        'IR_test': IR_test
    })

df_alphas = pd.DataFrame(results)

## 4. Evaluation Metric

An $\alpha$ that yields a portfolio with $k \le 50$, offering the lowest Tracking Error (within $10 \le k \le 50$ range).

In [5]:
# Filter results for 10 <= k <= 50 (or closest possible)
valid_portfolios = df_alphas[(df_alphas['k'] <= 50) & (df_alphas['k'] >= 10)]

if valid_portfolios.empty:
    chosen_row = df_alphas.iloc[0]
else:
    chosen_row = valid_portfolios.loc[valid_portfolios['TE_test'].idxmin()]

chosen_alpha = chosen_row['alpha']

# Refit the final portfolio with strict k<=50 limit
final_tickers, final_weights, final_k = fit_lasso_portfolio(X_train, y_train, alpha_val=chosen_alpha, max_k=50)

# Evaluate on Train
R_p_train = X_train[final_tickers] @ final_weights
final_TE_train, final_IR_train = evaluate_portfolio(R_p_train, y_train)

# Evaluate on Test
R_p_test = X_test[final_tickers] @ final_weights
final_TE_test, final_IR_test = evaluate_portfolio(R_p_test, y_test)


# 5. Final Portfolio Selection

In [6]:
print("FINAL PORTFOLIO SELECTION")
print(f"Chosen Alpha: {chosen_alpha:.2e}")
print(f"Final Count (k): {final_k}")
print(f"Train     -> TE: {final_TE_train:.6f} | IR: {final_IR_train:.4f}")
print(f"Test      -> TE: {final_TE_test:.6f} | IR: {final_IR_test:.4f}")
print(f"Selected Stocks: {', '.join(final_tickers)}")

FINAL PORTFOLIO SELECTION
Chosen Alpha: 4.50e-05
Final Count (k): 43
Train     -> TE: 0.008204 | IR: 0.0536
Test      -> TE: 0.004503 | IR: 0.0973
Selected Stocks: AAPL, AMD, AMP, AMZN, ANET, APA, APO, AVGO, BHF, BLK, BX, CBRE, CCL, CLF, CVNA, DXC, FCX, FIX, GOOGL, GPN, HWM, INTU, IQV, ISRG, IVZ, KLAC, LRCX, MCHP, META, MPWR, MSFT, NBR, NVDA, PAYC, PVH, SIG, SSP, SWK, SYF, TSLA, URI, VST, XYZ


## 6. Deliverables

### Plot 1: Sparsity vs. Tracking Error

In [7]:
# Plotting k vs TE
# Filter to plot 10 to 100
plot_data = df_alphas[(df_alphas['k'] >= 10) & (df_alphas['k'] <= 100)].sort_values('k')
if plot_data.empty:
    plot_data = df_alphas.sort_values('k')

fig = px.line(plot_data, x='k', y='TE_test', markers=True, 
              title='Sparsity (k) vs Tracking Error',
              labels={'k': 'Number of Selected Stocks (k)', 'TE_test': 'Tracking Error (TE)'})

# Add vertical line for chosen k
fig.add_vline(x=final_k, line_dash='dash', line_color='red', annotation_text=f'Chosen k={final_k}')
fig.update_layout(template='plotly_white')
fig.show()


### Plot 2: Cumulative Portfolio vs Benchmark Returns (on Test Data)

In [8]:
cum_R_p_test = R_p_test.cumsum()
cum_y_test = y_test.cumsum()

fig = go.Figure()
fig.add_trace(go.Scatter(x=cum_R_p_test.index, y=cum_R_p_test, mode='lines', name=f'Replicated Portfolio (k={final_k})', line=dict(color='green', width=2)))
fig.add_trace(go.Scatter(x=cum_y_test.index, y=cum_y_test, mode='lines', name='S&P 500 Benchmark', line=dict(color='black', width=2, dash='dash')))

fig.update_layout(
    title='Cumulative Returns Comparison (Holdout Period)',
    xaxis_title='Date',
    yaxis_title='Cumulative Log Return',
    template='plotly_white',
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)
fig.show()


### Plot 3: Sector Drift Analysis

In [9]:
portfolio_sectors = []
for ticker in final_tickers:
    yf_ticker = ticker.replace('-', '.')
    info = yf.Ticker(yf_ticker).info
    sector = info.get('sector')
    portfolio_sectors.append(sector)

# Aggregate Portfolio Sector Weights
df_weights = pd.DataFrame({'Ticker': final_tickers, 'Weight': final_weights, 'Sector': portfolio_sectors})
port_sector_weights = df_weights.groupby('Sector')['Weight'].sum()

# Approximate S&P 500 Sector Weights
sp500_approx_weights = {
    'Technology': 0.29,
    'Financial Services': 0.13,
    'Healthcare': 0.13,
    'Consumer Cyclical': 0.10,
    'Industrials': 0.08,
    'Communication Services': 0.08,
    'Consumer Defensive': 0.07,
    'Energy': 0.04,
    'Utilities': 0.03,
    'Real Estate': 0.02,
    'Basic Materials': 0.02,
}

# Normalize portfolio sectors terminology to closely match map if possible
port_sector_weights.index = [s.replace('Information Technology', 'Technology').replace('Health Care', 'Healthcare') for s in port_sector_weights.index]

# Align data for plotting
all_sectors = list(set(port_sector_weights.index).union(set(sp500_approx_weights.keys())))
all_sectors = [s for s in all_sectors if s != 'Unknown']

p_weights = [port_sector_weights.get(s, 0.0) for s in all_sectors]
b_weights = [sp500_approx_weights.get(s, 0.0) for s in all_sectors]

fig = go.Figure(data=[
    go.Bar(name='Replicated Portfolio', x=all_sectors, y=p_weights, marker_color='Green', opacity=0.7),
    go.Bar(name='S&P 500 Approx.', x=all_sectors, y=b_weights, marker_color='Red', opacity=0.7)
])
fig.update_layout(
    title='Sector Drift Analysis: Portfolio vs. Benchmark Weights',
    xaxis_title='Sector',
    yaxis_title='Weight',
    barmode='group',
    template='plotly_white'
)
fig.show()
